# Polar-is Query Demo

This notebook calls the implemented query functions:

* Timeseries
* Heatmap
* Find-time
* Find-area

We print the `QueryResult`, `GroupResult`, and `SourceInfo` of a result, and display the returned `xarray.Dataset` and plots when specified.

The executor also writes each returned group to its configured NetCDF output path.

## Functions

#### Driver: **`run_scenario(base_query, func_calls)`** - prints query results of all functions in `func_calls` of the given `base_query`

`find_project_root()` - determines the absolute directory path of Polar-is

`print_query_result()` - prints `QueryResult` information, prints group info individually

`plot_query_result()` - plots results of multiple groups

Helper functions:

* `sorted_coverage_cells()` - returns the bucket ID and the timestamp of cells
* `display_longitudes()` - converts longitudes from 0...360 to -180...180
* `projection_for_group()` - finds CRS for group
* `cell_polygon()` - make grid from flattened cells [WILL BE REPLACED WITH ORIGINAL x, y COORDINATES]
* `plot_spatial_values()` - plot heatmap; color edge red if find-area function
* `plot_group_result()` - plot one results of one group: for get-data, plots heatmap at the first timestamp


In [2]:
'''imports, find_project_root'''
from pathlib import Path
import sys

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import Normalize
from matplotlib.patches import Patch
import numpy as np
from IPython.display import display


def find_project_root():
    # This works when Jupyter starts either in the repository or beside this notebook.
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'config.yaml').is_file() and (candidate / 'storage').is_dir():
            return candidate
    raise RuntimeError('Start Jupyter from the polar-is repository or one of its subdirectories.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from storage.query_data.executor import execute_query

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/bean/Documents/school/research/current/polar-is


In [3]:
'''print_query_result()'''
def sorted_coverage_cells(cells):
    return [
        (bucket_id, timestamp.isoformat())
        for bucket_id, timestamp in sorted(cells, key=lambda item: (item[0], item[1]))
    ]


def print_query_result(result):
    print('QueryResult')
    print(f'  query_id: {result.query_id}')
    print(f'  function: {result.function}')
    print(f'  warnings: {list(result.warnings)}')
    print(f'  unmatched_miss_set: {sorted_coverage_cells(result.unmatched_miss_set)}')
    print(f'  groups: {len(result.groups)}')

    for number, group in enumerate(result.groups, start=1):
        print(f'\n  GroupResult {number}')
        print(f'\t    group_id: {group.group_id}')
        print(f'\t    file_path: {group.file_path}')
        print(f'\t    units: {group.units!r}')
        print(f'\t    hit_set: {sorted_coverage_cells(group.hit_set)}')
        print(f'\t    miss_set: {sorted_coverage_cells(group.miss_set)}')
        print(f'\t    warnings: {list(group.warnings)}')

        source = group.source
        print('    source:')
        print(f'      repository: {source.repository}')
        print(f'      dataset: {source.dataset}')
        print(f'      variable: {source.variable}')
        print(f'      additional_parameters: {source.additional_parameters}')
        print(f'      native_temporal_resolutions: {source.native_temporal_resolutions}')
        print(f'      native_spatial_resolutions: {source.native_spatial_resolutions}')
        print(f'      block_ids: {source.block_ids}')
        print('    data:')
        display(group.data)

In [5]:
'''plot_query_result()'''

GEOGRAPHIC_CRS = ccrs.PlateCarree()


def display_longitudes(longitudes):
    # Stored longitudes are 0..360; plotting at -180..180 is easier to read.
    return (np.asarray(longitudes) + 180.0) % 360.0 - 180.0


def projection_for_group(group):
    attrs = group.data[group.source.variable].attrs
    if attrs.get('GRIB_gridType') != 'lambert':
        return GEOGRAPHIC_CRS

    central_longitude = display_longitudes(
        float(attrs['GRIB_LoVInDegrees'])
    ).item()
    return ccrs.LambertConformal(
        central_longitude=central_longitude,
        central_latitude=float(attrs['GRIB_LaDInDegrees']),
        standard_parallels=(
            float(attrs['GRIB_Latin1InDegrees']),
            float(attrs['GRIB_Latin2InDegrees']),
        ),
    )


def cell_polygons(group, longitude, latitude):
    # Query results remain flattened cells. For plotting only, make one
    # rectangle around each center using the source grid's cell size.
    attrs = group.data[group.source.variable].attrs
    projection = projection_for_group(group)
    factor = float(group.data.attrs.get('coarseness_factor', 1))

    if attrs.get('GRIB_gridType') == 'lambert':
        centers = projection.transform_points(
            GEOGRAPHIC_CRS, longitude, latitude
        )[:, :2]
        x_center, y_center = centers[:, 0], centers[:, 1]
        half_width = float(attrs['GRIB_DxInMetres']) * factor / 2.0
        half_height = float(attrs['GRIB_DyInMetres']) * factor / 2.0
    else:
        x_center, y_center = longitude, latitude
        native_resolution = float(group.source.native_spatial_resolutions[0])
        half_width = native_resolution * factor / 2.0
        half_height = native_resolution * factor / 2.0

    vertices = np.stack(
        [
            np.column_stack((x_center - half_width, y_center - half_height)),
            np.column_stack((x_center + half_width, y_center - half_height)),
            np.column_stack((x_center + half_width, y_center + half_height)),
            np.column_stack((x_center - half_width, y_center + half_height)),
        ],
        axis=1,
    )
    return projection, vertices


def plot_spatial_values(
    group, values, title, cmap='viridis', colorbar_label=None, matches=None
):
    data = group.data
    longitude = display_longitudes(data['longitude'].values)
    latitude = np.asarray(data['latitude'].values)
    values = np.asarray(values)
    valid = np.isfinite(values) & np.isfinite(longitude) & np.isfinite(latitude)

    projection, vertices = cell_polygons(
        group, longitude[valid], latitude[valid]
    )
    fig, ax = plt.subplots(
        figsize=(8, 6), subplot_kw={'projection': projection}
    )
    cells = PolyCollection(
        vertices, array=values[valid], cmap=cmap, edgecolors='none',
        transform=projection,
    )
    ax.add_collection(cells)

    if valid.any():
        longitude_padding = max(np.ptp(longitude[valid]) * 0.03, 0.2)
        latitude_padding = max(np.ptp(latitude[valid]) * 0.03, 0.2)
        ax.set_extent(
            [
                longitude[valid].min() - longitude_padding,
                longitude[valid].max() + longitude_padding,
                max(-90.0, latitude[valid].min() - latitude_padding),
                min(90.0, latitude[valid].max() + latitude_padding),
            ],
            crs=GEOGRAPHIC_CRS,
        )
        colorbar = fig.colorbar(cells, ax=ax, format='%.3f')
        colorbar.set_label(colorbar_label or group.source.variable)

    if matches is not None:
        matched = valid & np.asarray(matches, dtype=bool)
        if matched.any():
            _, matched_vertices = cell_polygons(
                group, longitude[matched], latitude[matched]
            )
            outlines = PolyCollection(
                matched_vertices, facecolors='none', edgecolors='crimson',
                linewidths=0.8, transform=projection, zorder=2,
            )
            ax.add_collection(outlines)
            ax.legend(
                handles=[Patch(facecolor='none', edgecolor='crimson',
                               label='predicate matched')]
            )

    gridlines = ax.gridlines(draw_labels=True, alpha=0.25)
    gridlines.top_labels = False
    gridlines.right_labels = False
    ax.set_title(title)
    plt.show()


def plot_group_result(group, function, filter_value=None):
    data = group.data
    variable = group.source.variable
    label = variable if group.units is None else f'{variable} ({group.units})'
    source_label = f'{group.source.repository} / {group.source.dataset}'

    if function == 'get-data':
        if data.sizes.get('timestamp', 0) == 0:
            return
        first_time = data['timestamp'].values[0]
        plot_spatial_values(
            group,
            data[variable].isel(timestamp=0).values,
            f'get-data: {source_label} at {first_time}',
            colorbar_label=label,
        )

    elif function in {'timeseries', 'find-time'}:
        timestamps = data['timestamp'].values
        values = data[variable].values
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(timestamps, values, marker='.', label=variable)
        if function == 'find-time':
            matches = data['matches'].values.astype(bool)
            ax.scatter(
                timestamps[matches], values[matches], color='crimson',
                zorder=3, label='predicate matched'
            )
            ax.axhline(filter_value, color='crimson', linestyle='--', alpha=0.6,
                       label=f'filter value = {filter_value}')
        ax.set(xlabel='Time', ylabel=label, title=f'{function}: {source_label}')
        ax.grid(alpha=0.25)
        ax.legend()
        fig.autofmt_xdate()
        plt.show()

    elif function == 'heatmap':
        plot_spatial_values(
            group, data[variable].values, f'heatmap: {source_label}',
            colorbar_label=label,
        )

    elif function == 'find-area':
        plot_spatial_values(
            group, data[variable].values,
            f'find-area: {source_label}; filter value = {filter_value}',
            colorbar_label=label, matches=data['matches'].values,
        )


def plot_query_result(result, filter_value=None):
    if not result.groups:
        print('No group data was returned, so there is nothing to plot.')
        return
    for group in result.groups:
        plot_group_result(group, result.function, filter_value=filter_value)

In [6]:
'''run_scenario'''
def run_scenario(base_query, func_calls):
    results = {}
    for function, function_fields in func_calls:
        query = {**base_query, 'function': function, **function_fields}
        print('\n' + '=' * 100)
        print(f'{function}')
        print('=' * 100)
        print('Query input:')
        display(query)

        result = execute_query(query)
        print_query_result(result)
        plot_query_result(result, filter_value=function_fields.get('filter_value'))
        results[function] = result
    return results

## Define `func_calls`

In [ ]:
FILTER_VAL_TIME = 278.5
FILTER_VAL_SPACE = 290.0

FUNCTION_CALLS = (
    ('get-data', {}),
    ('timeseries', {}),
    ('heatmap', {}),
    ('find-time', {'predicate': 'gt', 'filter_value': FILTER_VALUE}),
    ('find-area', {'predicate': 'gt', 'filter_value': FILTER_VALUE}),
)

## Define `base_query`

In [7]:
COMMON_QUERY = {
    'variable': 'sea_surface_temperature',
    'time_start': '2018-01',
    'time_end': '2018-01',
    'coarseness_factor': 1,
    'time_unit': 'Day',
    'aggregation_method': 'mean',
}

NO_MISSES = {
    **COMMON_QUERY,
    'region': {'west': 60.0, 'east': -35.0, 'south': 20.0, 'north': 25.0},
    'repository': 'copernicusclimatedatastore',
    'dataset': 'era5_single_level',
}

SOME_MISSES = {
    **COMMON_QUERY,
    'region': {'west': 60.0, 'east': 179.999, 'south': 65.0, 'north': 75.0},
    'repository': 'copernicusclimatedatastore',
    'dataset': 'carra_height',
    'additional_parameters': {'height': '15m'},
}

ALL_MISSES = {
    **COMMON_QUERY,
    'region': {'west': 10.0, 'east': 20.0, 'south': -20.0, 'north': -10.0},
    'repository': 'copernicusclimatedatastore',
    'dataset': 'carra_height',
    'additional_parameters': {'height': '15m'},
}

# to display, use display({'str': base_query})

## Queries

### Scenarios: no misses, some misses, all misses